# 08 - Target diagnostics

**What this notebook is for.** Every number, table and figure that this notebook
contributes to the thesis is produced here, from the project's own data. Nothing is copied in by hand and nothing depends on a
one-off command run outside the repository. If a value cannot be regenerated by
running this notebook top to bottom, it does not go in the thesis.

**Scope.** The notebook answers one central question - *what does the
NBR-derived recovery-gap indicator actually measure, and are its land-cover
differences a property of post-fire recovery or of the indicator's
construction?* - plus the threshold, metric and reproducibility questions raised
alongside it.

**What it does not do.** Nothing here retrains a model, recomputes a stored
target, or modifies the pipeline. Every model output used is read from the
artefacts written by Notebook 05. The analytical targets are read from
`targets_aux.npz` exactly as Notebook 04 wrote them; where the notebook
recomputes a target it does so only to *verify* the stored one or to construct
an explicitly-labelled alternative.

**Runtime.** About 3-6 minutes. Peak memory ~3 GB (the three extracted spectral
arrays are memory-mapped and read in slices).

---

## Contents

| § | Analysis | Needs |
|---|---|---|
| 1 | Environment, data inventory and integrity checks | core |
| 2 | Code-provenance assertions (thesis-vs-code facts) | notebook sources |
| 3 | Baseline audit - which reference period each indicator uses | - |
| 4 | Analytical targets by land-cover class, before any model | core |
| 5 | Fire-timing controls | core |
| 6 | Boundary decomposition - how much of a class mean is the ceiling | core |
| 7 | Normalisation decomposition - numerator vs denominator | NBR |
| 8 | Event-aligned post-fire NBR response, seasonality cancelled | NBR |
| 9 | Does the indicator track measured recovery? | NBR |
| 10 | Alternative formulations, including an independent index | NBR, NDVI |
| 11 | Corrected recovery-window sensitivity | NBR |
| 12 | Persistence threshold sensitivity | NBR z-score |
| 13 | Model versus land cover - three explanations separated | predictions |
| 14 | Where the reconstruction R² comes from | predictions |
| 15 | Classification metrics under class imbalance | predictions |
| 16 | Display-mask semantics and the severity criterion | 30 m rasters |
| 17 | Which land-cover definition Table 11 uses | predictions |
| 18 | Result registry - thesis result → cell → output file | - |

"core" = `targets_aux.npz`, `X_scaled_meta.csv`, `y_burned.npy`.


## 1. Environment, data inventory and integrity checks

**Question.** Are the inputs this notebook depends on present, and are they the ones the thesis reports on?

**Why.** Several results below are only meaningful if the arrays are the exact artefacts written by Notebooks 04 and 05. The large spectral arrays in particular are *derived* files produced outside this notebook (see the provenance note in the next cell), so their integrity has to be established before anything is computed from them.

**Data.** Everything under `data/processed/` and `models/main/`, plus the 30 m rasters under `outputs/inference_full_roi/`.

**Metric.** File presence, shape, dtype and SHA-256 of the first and last 16 MiB of each large array (a full hash of 263 MB files is slow and adds nothing). Sections that need a missing input are skipped with an explicit notice rather than failing.

**Reading the output.** A table of every input with its size and fingerprint. Record these fingerprints if the thesis is submitted with a data statement.

**Relates to.** Thesis Reproducibility requirement for all new results.

In [ ]:
import hashlib
import json
import os
import platform
import sys
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Resolve the project root whether the notebook is run from notebooks/ or the root.
ROOT = os.environ.get("THESIS_ROOT")
if not ROOT:
    here = os.path.abspath(os.getcwd())
    ROOT = os.path.dirname(here) if os.path.basename(here) == "notebooks" else here
ROOT = os.path.expanduser(ROOT)

PROC = os.path.join(ROOT, "data", "processed")
MAIN = os.path.join(ROOT, "models", "main")
RAST = os.path.join(ROOT, "outputs", "inference_full_roi")
TAB = os.path.join(ROOT, "outputs", "tables")
FIG = os.path.join(ROOT, "outputs", "figures")
NBDIR = os.path.join(ROOT, "notebooks")
os.makedirs(TAB, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
print("thesis root :", ROOT)
print("run (UTC)   :", RUN_STAMP)
print("python      :", sys.version.split()[0], "|", platform.platform())
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", stats.__name__ and __import__("scipy").__version__)

In [ ]:
# ---- shared constants -------------------------------------------------------
LC_NAMES = {10: "forest", 11: "orchards", 20: "shrubland", 30: "grassland",
            40: "cropland", 41: "covered agriculture", 50: "built-up",
            60: "bare", 61: "fallow", 80: "water"}
MIN_N = 500          # classes below this are computed but flagged as unreliable
EPS = 1e-6           # the epsilon in the recovery-gap denominator (Notebook 04)
BURN_THRESHOLD = 0.502   # the F1-optimised operating threshold (Notebook 05/07b)
SEED = 42
RNG = np.random.default_rng(SEED)

# Output naming: every file this notebook writes is prefixed nb08_ so its
# provenance is visible from the filename alone.
PREFIX = "nb08_"
REGISTRY = []        # populated by save_table / save_figure, emitted in section 18


def _fingerprint(path, edge=16 * 1024 * 1024):
    """SHA-256 over the first and last `edge` bytes - enough to detect a swap."""
    h = hashlib.sha256()
    size = os.path.getsize(path)
    with open(path, "rb") as fh:
        h.update(fh.read(min(edge, size)))
        if size > edge:
            fh.seek(max(0, size - edge))
            h.update(fh.read(edge))
    return h.hexdigest()[:16], size


def _markdown_table(df, fmt="{:.4f}"):
    """Markdown writer with no dependency on `tabulate`."""
    def cell(v):
        if isinstance(v, (float, np.floating)):
            return "" if not np.isfinite(v) else fmt.format(v)
        return str(v)
    cols = list(df.columns)
    out = ["| " + " | ".join(cols) + " |",
           "|" + "|".join("---" for _ in cols) + "|"]
    for _, row in df.iterrows():
        out.append("| " + " | ".join(cell(row[c]) for c in cols) + " |")
    return "\n".join(out) + "\n"


def save_table(df, stem, caption, section):
    """Write a table as CSV + Markdown and register it for section 18."""
    name = PREFIX + stem
    csv_path = os.path.join(TAB, name + ".csv")
    md_path = os.path.join(TAB, name + ".md")
    df.to_csv(csv_path, index=False)
    with open(md_path, "w") as fh:
        fh.write("<!-- generated by notebooks/08_target_diagnostics.ipynb §" +
                 str(section) + " on " + RUN_STAMP + " -->\n\n")
        fh.write(caption.rstrip() + "\n\n")
        fh.write(_markdown_table(df))
    REGISTRY.append(dict(kind="table", section=section, stem=name,
                         caption=caption, files="outputs/tables/" + name + ".csv, .md"))
    print("  -> outputs/tables/" + name + ".csv  (+ .md)")
    return df


def save_figure(fig, stem, caption, section, dpi=300):
    name = PREFIX + stem
    path = os.path.join(FIG, name + ".png")
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    REGISTRY.append(dict(kind="figure", section=section, stem=name,
                         caption=caption, files="outputs/figures/" + name + ".png"))
    print("  -> outputs/figures/" + name + ".png")
    return path


def note(*msg):
    print("   ", *msg)

In [ ]:
# ---- inventory --------------------------------------------------------------
INPUTS = {
    "targets_aux.npz":       os.path.join(PROC, "targets_aux.npz"),
    "X_scaled_meta.csv":     os.path.join(PROC, "X_scaled_meta.csv"),
    "y_burned.npy":          os.path.join(PROC, "y_burned.npy"),
    "splits.json":           os.path.join(PROC, "splits.json"),
    "predictions_test.npz":  os.path.join(MAIN, "predictions_test.npz"),
    "results_main.json":     os.path.join(MAIN, "results_main.json"),
    "nbr_raw.npy":           os.path.join(PROC, "nbr_raw.npy"),
    "nbr_zscore_raw.npy":    os.path.join(PROC, "nbr_zscore_raw.npy"),
    "ndvi_raw.npy":          os.path.join(PROC, "ndvi_raw.npy"),
    "severity_30m.tif":      os.path.join(RAST, "severity_30m.tif"),
    "burned_prob_30m.tif":   os.path.join(RAST, "burned_prob_30m.tif"),
    "model_recurrence_30m.tif": os.path.join(RAST, "model_recurrence_30m.tif"),
    "recurrence_30m.tif":    os.path.join(RAST, "recurrence_30m.tif"),
}

rows = []
for label, path in INPUTS.items():
    if os.path.exists(path):
        fp, size = _fingerprint(path)
        rows.append(dict(input=label, present=True, size_MB=round(size / 1e6, 1),
                         sha256_head_tail=fp))
    else:
        rows.append(dict(input=label, present=False, size_MB=np.nan,
                         sha256_head_tail=""))
inventory = pd.DataFrame(rows)

HAVE = {k: bool(inventory.loc[inventory["input"] == k, "present"].iloc[0])
        for k in INPUTS}
HAVE_SPECTRAL = HAVE["nbr_raw.npy"] and HAVE["nbr_zscore_raw.npy"]
HAVE_NDVI = HAVE["ndvi_raw.npy"]
HAVE_PREDS = HAVE["predictions_test.npz"]
HAVE_RASTERS = all(HAVE[k] for k in
                   ("severity_30m.tif", "burned_prob_30m.tif",
                    "model_recurrence_30m.tif", "recurrence_30m.tif"))

display(inventory)
print()
print("spectral arrays :", HAVE_SPECTRAL, "| NDVI:", HAVE_NDVI)
print("model predictions:", HAVE_PREDS, "| 30 m rasters:", HAVE_RASTERS)
save_table(inventory, "t00_input_inventory",
           "Inputs consumed by this notebook, with a head+tail SHA-256 fingerprint "
           "(first and last 16 MiB) for each file.", section=1)

### Provenance of the three extracted spectral arrays

`nbr_raw.npy`, `nbr_zscore_raw.npy` and `ndvi_raw.npy` are **derived files**, not
pipeline outputs. They are not produced by this notebook because they come from
`data/processed/X_scaled.npy`, a 13.7 GB tensor that is impractical to keep on
an analysis machine.

Each is one column of that tensor with the training-set standardisation
inverted:

```
raw[:, t] = X_scaled[:, t, col] * scaler.scale_[pos] + scaler.mean_[pos]
```

where `col` is the feature's index in `feature_names.json` and `pos` its
position within `scaler["spectral_cols"]`. `StandardScaler` is exactly
invertible, so this recovers the pre-scaling values that Notebook 04 used when
it built the targets.

Two scripts produce them, and both end with the same self-check - they rebuild
`recovery_gap`, `severity` and `persistence` from the extracted columns and
compare against the stored `targets_aux.npz`:

| Script | Use when |
|---|---|
| `gee/extract_nbr_arrays.py` | `X_scaled.npy` is on the local disk. Memory-mapped, chunked; peak memory ~250 MB. |
| `gee/stream_extract_nbr.py` | `X_scaled.npy` is reachable only by URL. HTTP range requests, resumable; never writes the 13.7 GB file. |

```bash
python gee/extract_nbr_arrays.py
```

Section 1b below repeats the verification independently, so this notebook does
not have to trust the extraction scripts' own report.


### 1b. Independent verification of the extracted arrays

**Question.** Do the extracted spectral columns reproduce the stored analytical targets?

**Why.** Sections 7-12 recompute the recovery gap under alternative definitions. Those alternatives are only comparable to the published indicator if the inputs they are built from are the same ones Notebook 04 used. This is the check that licenses everything downstream.

**Data.** `nbr_raw.npy`, `nbr_zscore_raw.npy`, `y_burned.npy`, `targets_aux.npz`.

**Metric.** Recompute all three targets with the exact Notebook 04 formulae and compare against the stored arrays: max absolute difference, share of pixels agreeing within 1e-2, and Pearson correlation.\n\nA per-pixel tolerance of 1e-4 is *not* the right bar. `X_scaled` stores standardised float32, so inverting it costs a few ULPs; the recovery gap then divides by |baseline NBR|, whose sample mean is only ~0.085, which amplifies that error - most where the baseline is smallest. Section 7 quantifies this conditioning explicitly. What matters downstream is rank agreement and class means, both of which are checked here.

**Reading the output.** PASS means the extracted arrays are the ones Notebook 04 used, to a precision far finer than any effect analysed below.

**Relates to.** Thesis Reproducibility requirement; also supplies the numerical-conditioning observation used in §7 and in the thesis limitations.

In [ ]:
targets = dict(np.load(INPUTS["targets_aux.npz"]))
meta = pd.read_csv(INPUTS["X_scaled_meta.csv"])
y_burned = np.load(INPUTS["y_burned.npy"], mmap_mode="r")

N_PIX, N_MONTHS = y_burned.shape
LC_ALL = meta["lc_dominant"].to_numpy()
BURNED = targets["recurrence"] > 0          # pixel had >=1 documented fire month
N_BURNED = int(BURNED.sum())

print("pixels in sample      :", format(N_PIX, ","))
print("months per pixel      :", N_MONTHS, "(Jan 2017 - Dec 2025)")
print("burned pixels         :", format(N_BURNED, ","))
assert N_PIX == len(meta) == len(targets["recovery_gap"])

if HAVE_SPECTRAL:
    nbr = np.load(INPUTS["nbr_raw.npy"], mmap_mode="r")
    nbr_z = np.load(INPUTS["nbr_zscore_raw.npy"], mmap_mode="r")
    assert nbr.shape == (N_PIX, N_MONTHS), nbr.shape

    yb_full = np.asarray(y_burned, dtype=np.float32)

    # recovery_gap: Notebook 04 cell 12, vectorised identically
    b_all = np.nanmean(np.asarray(nbr[:, :12], dtype=np.float64), axis=1)
    r_all = np.nanmean(np.asarray(nbr[:, -12:], dtype=np.float64), axis=1)
    rg_chk = np.zeros(N_PIX, dtype=np.float32)
    rg_chk[BURNED] = np.clip((b_all - r_all) / (np.abs(b_all) + EPS), 0, 1)[BURNED]

    # severity: mean of -z over burned months, /5, clipped. Vectorised as a
    # masked mean, which is exactly mean(-z[burned months]).
    zf = np.asarray(nbr_z, dtype=np.float64)
    n_fire = yb_full.sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        sev_chk = np.clip((-(zf * yb_full).sum(axis=1) / np.where(n_fire > 0, n_fire, 1)) / 5.0, 0, 1)
    sev_chk = np.where(BURNED, sev_chk, 0.0).astype(np.float32)

    # persistence: fraction of months from the last fire month onward with z < -0.5
    idx_m = np.arange(N_MONTHS)[None, :]
    last_fire = np.where(BURNED,
                         N_MONTHS - 1 - np.argmax(np.asarray(y_burned)[:, ::-1] > 0, axis=1),
                         -1)
    post = idx_m >= last_fire[:, None]
    n_post = post.sum(axis=1)
    per_chk = np.where((n_post > 1) & BURNED,
                       ((zf < -0.5) & post).sum(axis=1) / np.maximum(n_post, 1),
                       0.0).astype(np.float32)
    del zf, post

    rows = []
    ok = True
    for name, got in (("recovery_gap", rg_chk), ("severity", sev_chk),
                      ("persistence", per_chk)):
        want = targets[name]
        diff = np.abs(got - want)
        corr = float(np.corrcoef(got, want)[0, 1])
        within = float(np.mean(diff < 1e-2))
        passed = (corr > 0.9999) and (within > 0.999)
        ok &= passed
        rows.append(dict(target=name, max_abs_diff=float(diff.max()),
                         share_within_1e2=within, pearson=corr, passed=passed))
    verification = pd.DataFrame(rows)
    display(verification)
    print("\nVERIFICATION:", "PASS" if ok else "FAIL")
    assert ok, "extracted arrays do not reproduce the stored targets — stop here"
    save_table(verification, "t01_extraction_verification",
               "Stored analytical targets rebuilt from the extracted spectral "
               "columns using the Notebook 04 formulae.", section="1b")
else:
    print("Spectral arrays absent — sections 7-12 will be skipped.")
    print("Run notebooks/extract_nbr_arrays.py (or stream_extract_nbr.py) first.")

## 2. Code-provenance assertions

**Question.** Do the thesis's statements about its own method match the code that ran?

**Why.** Several statements the thesis makes are claims *about the code* rather than about the data - that the composites are means and not medians, that the display mask is a maximum over months, and so on. Those claims must be as reproducible as the numbers. This cell derives each one from the notebook sources rather than from anyone's reading of them.

**Data.** The `.ipynb` sources of Notebooks 01, 04 and 07b, and `model_defs.py`.

**Metric.** Substring assertions over the concatenated code cells of each notebook. Each check reports the fact, what the thesis currently says, and whether the code supports it.

**Reading the output.** `code_supports_thesis = False` marks a statement in the thesis that the code contradicts. Each such case is resolved in the thesis text.


In [ ]:
def _nb_source(filename):
    """Concatenate the code cells of a notebook (or return a .py file whole)."""
    path = os.path.join(NBDIR, filename)
    if not os.path.exists(path):
        path = os.path.join(ROOT, filename)
    if not os.path.exists(path):
        return None
    if path.endswith(".py"):
        return open(path).read()
    try:
        nb = json.load(open(path))
    except json.JSONDecodeError:
        return open(path).read()          # a .py mislabelled as .ipynb
    return "\n".join("".join(c.get("source", []))
                      for c in nb.get("cells", []) if c.get("cell_type") == "code")

SRC = {name: _nb_source(name) for name in
       ("01_data_to_gee.ipynb", "04_data_prep.ipynb",
        "07b_inference_full_roi.ipynb", "model_defs.py")}

checks = [
    ("monthly composites use mean, not median", "01_data_to_gee.ipynb",
     lambda s: ("ic_ref.select(BANDS).mean()" in s) and ("monthly_mean_image" in s),
     "thesis §3.2.2 and Table 1 say 'monthly median composites'",
     "register #7"),
    ("no explicit resampling of the 20 m bands", "01_data_to_gee.ipynb",
     lambda s: (".resample(" not in s) and (".reproject(" not in s),
     "thesis §3.2.2 says the 20 m bands were 'aggregated'/'resampled', method unstated; "
     "absence of a call means GEE's default nearest neighbour",
     "register #8, C315"),
    ("recovery-gap baseline is the first 12 months", "04_data_prep.ipynb",
     lambda s: "nbr_raw[i, :baseline_months]" in s,
     "thesis §5.4 and §5.6 attribute recovery-gap behaviour to the '2016-2017 baseline'",
     "register #4"),
    ("recovery gap is zero for unburned pixels", "04_data_prep.ipynb",
     lambda s: "if len(burned_months) == 0:" in s,
     "matches thesis §3.2.7",
     "no discrepancy"),
    ("persistence includes the last fire month", "04_data_prep.ipynb",
     lambda s: "nbr_z[last_fire:]" in s,
     "matches the corrected Eq 3.13 (m >= m*)",
     "no discrepancy"),
    ("burned_prob raster is a max over months", "07b_inference_full_roi.ipynb",
     lambda s: "burned_probs.max(axis=1)" in s,
     "thesis §3.4.6/§4.11 describe it as 'the estimated burned probability', "
     "implying the per-month value",
     "register #2"),
    ("model recurrence counts months above threshold", "07b_inference_full_roi.ipynb",
     lambda s: "(burned_probs > BURN_THRESHOLD).sum(axis=1) / 5.0" in s,
     "matches thesis §3.2.7",
     "no discrepancy"),
    ("auxiliary loss is divided by 3", "model_defs.py",
     lambda s: "(loss_sev + loss_per + loss_rec) / 3" in s,
     "thesis Eq 3.19 writes lambda_aux * mean(L_sev + L_per + L_rec), which is ambiguous",
     "register #12"),
]

rows = []
for fact, src_name, test, thesis_says, maps_to in checks:
    s = SRC.get(src_name)
    rows.append(dict(fact=fact, source=src_name,
                     code_confirms=(None if s is None else bool(test(s))),
                     thesis_says=thesis_says, maps_to=maps_to))
provenance = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 70)
display(provenance)
missing = provenance["code_confirms"].isna().sum()
if missing:
    print("\n", missing, "check(s) skipped — source not found")
save_table(provenance, "t02_code_provenance_checks",
           "Statements the thesis makes about its own method, tested directly "
           "against the notebook sources that produced the results.", section=2)

## 3. Baseline audit

**Question.** Which reference period does each analytical indicator actually use?

**Why.** The thesis treats 'the 2016-2017 baseline' as a single object. It is not. Severity and persistence are built from the z-score channel, which uses a per-calendar-month baseline computed in Google Earth Engine from the 2016 and 2017 composites. The recovery gap uses something different: the pixel's own mean raw NBR over calendar 2017, computed in Notebook 04. Conflating the two attributes recovery-gap behaviour to the wrong baseline.

**Data.** Read out of the code rather than computed: Notebook 01 Step 6 (GEE baseline assets) and Notebook 04 cell 12 (recovery gap).

**Metric.** None - this is a documentation table. The `n_obs_per_month` column is the number of observations behind each reference statistic, which is what makes the z-score baseline fragile: two images per calendar month, reduced with `ee.Reducer.stdDev()`, i.e. a population standard deviation over n = 2.

**Reading the output.** The recovery gap is the only indicator not using the GEE baseline. Any statement about '2016-2017' that is offered to explain recovery-gap behaviour is about the wrong quantity.

**Relates to.** Thesis §3.2.7, §5.4, §5.6.

In [ ]:
baseline_audit = pd.DataFrame([
    dict(indicator="severity",
         formula="clip(mean(-z_NBR over fire months) / 5, 0, 1)",
         reference_period="2016-2017, per calendar month",
         computed_in="Notebook 01 Step 6 -> NBR_zscore feature",
         n_obs_per_month=2),
    dict(indicator="persistence",
         formula="mean(1[z_NBR < -0.5]) from the last fire month onward",
         reference_period="2016-2017, per calendar month",
         computed_in="Notebook 01 Step 6 -> NBR_zscore feature",
         n_obs_per_month=2),
    dict(indicator="recovery gap",
         formula="clip((b - r) / (|b| + 1e-6), 0, 1),  b = mean NBR months 0-11, r = months 96-107",
         reference_period="Jan-Dec 2017 only, the pixel's own mean raw NBR",
         computed_in="Notebook 04 cell 12, nbr_raw[:, :12]",
         n_obs_per_month=1),
    dict(indicator="anomaly and z-score features",
         formula="(x - mu_month) / sigma_month",
         reference_period="2016-2017, per calendar month",
         computed_in="Notebook 01 Step 6",
         n_obs_per_month=2),
])
display(baseline_audit)
save_table(baseline_audit, "t03_baseline_audit",
           "Reference period used by each analytical indicator. The recovery gap "
           "is the only one that does not use the 2016-2017 GEE baseline.",
           section=3)

# The practical consequence of an n=2 standard deviation, read off the scaler.
scaler_path = os.path.join(PROC, "scaler.pkl")
names_path = os.path.join(PROC, "feature_names.json")
if os.path.exists(scaler_path) and os.path.exists(names_path):
    import pickle
    bundle = pickle.load(open(scaler_path, "rb"))
    fnames = json.load(open(names_path))
    sc, spec_cols = bundle["scaler"], bundle["spectral_cols"]
    rows = []
    for nm in ("NBR", "NDVI", "NBR_zscore", "NDVI_zscore"):
        if nm in fnames and fnames.index(nm) in spec_cols:
            pos = spec_cols.index(fnames.index(nm))
            rows.append(dict(feature=nm, train_mean=float(sc.mean_[pos]),
                             train_scale=float(sc.scale_[pos])))
    zstats = pd.DataFrame(rows)
    display(zstats)
    note("The z-score channels are standardised with scales three to five orders of")
    note("magnitude larger than the raw indices, because the n=2 baseline sigma")
    note("approaches zero in places. Severity and persistence both depend on this.")
    save_table(zstats, "t03b_scaler_statistics",
               "Training-set standardisation statistics for the raw and z-scored "
               "index channels, from data/processed/scaler.pkl.", section=3)

## 4. Analytical targets by land-cover class, before any model

**Question.** Are the land-cover differences in the recovery gap present in the analytical target itself, independently of the model?

**Why.** The analytical recovery-gap target and the post-fire NBR curves are presented by land-cover class before the model is applied. If the differences are already in the target, then the model reproducing them carries no ecological information.

**Data.** `targets_aux.npz` (analytical targets as written by Notebook 04), `X_scaled_meta.csv` (`lc_dominant`, the modal current-year class across the 108 months), `y_burned.npy` (fire months). Burned pixels only, i.e. those with at least one documented fire month, which is the population the regression heads were trained on.

**Metric.** Class means, medians and standard deviations, plus the share of pixels at each bound. `pct_at_0` is the share whose end-of-record NBR is at or above the 2017 baseline (nothing outstanding); `pct_at_1` is the share clipped at the ceiling. `rg_mean_interior` averages only the pixels strictly inside (0, 1).

**Reading the output.** A large spread of class means, and a strongly bimodal distribution in the classes that score highest. Note that `rg_mean_interior` is *not* a counterfactual - it conditions on a non-random subset - so it must not be reported as 'what the gap would be without clipping'. Section 6 gives the decomposition that is exact.

**Relates to.** Thesis §4.4 and §5.4.

In [ ]:
yb = np.asarray(y_burned[BURNED])            # (n_burned, 108) fire labels
lcb = LC_ALL[BURNED]
rg_b = targets["recovery_gap"][BURNED]
sev_b = targets["severity"][BURNED]
per_b = targets["persistence"][BURNED]

first_fire = np.argmax(yb > 0, axis=1)
last_fire_b = N_MONTHS - 1 - np.argmax(yb[:, ::-1] > 0, axis=1)

rows = []
for code, name in LC_NAMES.items():
    m = lcb == code
    if not m.any():
        continue
    rg = rg_b[m]
    interior = (rg > EPS) & (rg < 1 - EPS)
    rows.append(dict(
        code=code, land_cover=name, n=int(m.sum()),
        rg_mean=float(rg.mean()), rg_median=float(np.median(rg)),
        rg_sd=float(rg.std()),
        pct_at_0=100 * float((rg <= EPS).mean()),
        pct_at_1=100 * float((rg >= 1 - EPS).mean()),
        rg_mean_interior=float(rg[interior].mean()) if interior.any() else np.nan,
        severity=float(sev_b[m].mean()), persistence=float(per_b[m].mean()),
        mean_first_fire_year=float((2017 + first_fire[m] // 12).mean()),
        mean_post_fire_months=float((N_MONTHS - 1 - last_fire_b[m]).mean()),
        pct_first_fire_in_2017=100 * float((first_fire[m] < 12).mean()),
        below_min_n=int(m.sum()) < MIN_N))
targets_by_lc = pd.DataFrame(rows).sort_values("rg_mean", ascending=False)
display(targets_by_lc.drop(columns=["code"]))

note("burned pixels:", format(N_BURNED, ","))
note("share of burned pixels whose FIRST fire falls inside the 2017 baseline window:",
     "{:.2f}%".format(100 * float((first_fire < 12).mean())),
     "(n = {:,})".format(int((first_fire < 12).sum())))
save_table(targets_by_lc, "t04_targets_by_landcover",
           "Analytical targets by dominant land-cover class, burned pixels only. "
           "'pct_at_0' is the share whose end-of-record NBR is at or above the "
           "2017 baseline; 'pct_at_1' the share clipped at the upper bound. "
           "Classes with n < 500 are flagged in 'below_min_n'.", section=4)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300,
                            "font.size": 9, "axes.grid": True,
                            "grid.alpha": 0.25, "axes.spines.top": False,
                            "axes.spines.right": False})

SHOW = [30, 61, 20, 40, 60, 10, 11]          # classes with n >= 500, ordered by rg
CMAP = plt.get_cmap("tab10")
COLOR = {c: CMAP(i) for i, c in enumerate(SHOW)}

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True)
for ax, key, title in zip(axes, ["recovery_gap", "severity", "persistence"],
                          ["Recovery gap", "Severity", "Persistence"]):
    data = [targets[key][BURNED][lcb == c] for c in SHOW]
    ax.violinplot(data, showmedians=True, widths=0.85)
    ax.set_xticks(range(1, len(SHOW) + 1))
    ax.set_xticklabels([LC_NAMES[c] for c in SHOW], rotation=45, ha="right", fontsize=8)
    ax.set_title(title + " (analytical target)", fontsize=10)
axes[0].set_ylabel("indicator value")
fig.suptitle("Analytical targets by land cover, burned pixels — before any model",
             fontsize=11)
fig.tight_layout()
save_figure(fig, "f01_targets_by_landcover",
            "Distribution of the three analytical targets by land-cover class on "
            "burned pixels. Recovery gap and severity are strongly bimodal with "
            "mass at both bounds; persistence is not.", section=4)
plt.show()

## 5. Fire-timing controls

**Question.** Could the land-cover ordering be an artefact of different land covers having burned at different times?

**Why.** The recovery gap compares fixed calendar windows - 2017 against 2025 - regardless of when a pixel burned. A pixel that burned in 2018 has seven years to recover before the recent window; one that burned in 2024 has one. If land-cover classes differ systematically in when they burned, that alone could produce the ordering. This is the most obvious rival explanation and has to be eliminated before any other is entertained.

**Data.** `y_burned.npy` for the month of each pixel's last documented fire; `targets_aux.npz` for the target.

**Metric.** Mean recovery-gap target within each (land cover × last-fire-year) cell. Cells with fewer than 100 pixels are suppressed as unreliable.

**Reading the output.** If the ordering is a timing artefact, it should weaken or reverse within year cohorts. If it holds in every cohort, timing is not the explanation.

**Relates to.** Thesis §5.4.

In [ ]:
cohort = pd.DataFrame({
    "land_cover": [LC_NAMES.get(c, str(c)) for c in lcb],
    "last_fire_year": 2017 + last_fire_b // 12,
    "recovery_gap": rg_b,
})
piv = cohort.pivot_table(index="land_cover", columns="last_fire_year",
                         values="recovery_gap", aggfunc="mean")
cnt = cohort.pivot_table(index="land_cover", columns="last_fire_year",
                         values="recovery_gap", aggfunc="size")
piv_masked = piv.where(cnt >= 100)
display(piv_masked.round(3))
note("cells with fewer than 100 pixels are suppressed")

rng_by_class = piv_masked.agg(["min", "max"], axis=1)
rng_by_class.columns = ["min_across_cohorts", "max_across_cohorts"]
rng_by_class = rng_by_class.join(targets_by_lc.set_index("land_cover")["rg_mean"])
display(rng_by_class.round(3))

out = piv_masked.reset_index()
out.columns = [str(c) for c in out.columns]
save_table(out, "t05_recovery_gap_by_lc_and_fire_year",
           "Mean analytical recovery gap by land-cover class and year of last "
           "documented fire. Cells with n < 100 are blank. The class ordering "
           "holding within every cohort rules out fire timing as its cause.",
           section=5)

## 6. Boundary decomposition - how much of a class mean is the ceiling?

**Question.** How much of each class's mean indicator value is contributed by pixels pinned at the upper bound rather than by graded variation?

**Why.** Both the recovery gap and severity are bounded to [0, 1] and both are strongly bimodal (§4). For a bounded variable the class mean is partly a *count of boundary membership*, not an average of a graded quantity. If one class saturates far more often than another, comparing their means compares saturation rates. The thesis interprets these means as graded differences.

**Data.** `targets_aux.npz` only. No spectral arrays needed, so this runs even without the extraction step.

**Metric.** An exact identity. Since pixels at zero contribute nothing,\n\n```\nclass mean = P(at 1) · 1  +  P(interior) · mean(interior)\n```\n\n`share_of_mean_from_ceiling = P(at 1) / class mean`. This is arithmetic, not a model, and it is exact.

**Reading the output.** A high `share_of_mean_from_ceiling` means the class mean is largely a saturation count. Comparing `mean_interior` across classes shows how much separation survives among pixels that are still varying - though, as in §4, that is a conditional subset and not a counterfactual.

**Relates to.** Thesis §4.4 (both the recovery-gap and the severity orderings).

In [ ]:
def boundary_decomposition(values, classes, label):
    rows = []
    for code, name in LC_NAMES.items():
        m = classes == code
        if m.sum() < MIN_N:
            continue
        v = values[m]
        at1 = v >= 1 - EPS
        at0 = v <= EPS
        interior = ~at1 & ~at0
        mean = float(v.mean())
        mi = float(v[interior].mean()) if interior.any() else np.nan
        rows.append(dict(
            indicator=label, land_cover=name, n=int(m.sum()), class_mean=mean,
            pct_at_0=100 * float(at0.mean()), pct_at_1=100 * float(at1.mean()),
            pct_interior=100 * float(interior.mean()), mean_interior=mi,
            contrib_ceiling=float(at1.mean()),
            contrib_interior=float(interior.mean() * mi) if interior.any() else np.nan,
            share_of_mean_from_ceiling=100 * float(at1.mean()) / mean if mean > 0 else np.nan))
    df = pd.DataFrame(rows).sort_values("class_mean", ascending=False)
    # the identity must hold exactly
    recomposed = df["contrib_ceiling"] + df["contrib_interior"]
    assert np.allclose(recomposed, df["class_mean"], atol=1e-9), "identity failed"
    return df

bd_rg = boundary_decomposition(rg_b, lcb, "recovery gap")
bd_sev = boundary_decomposition(sev_b, lcb, "severity")
bd_per = boundary_decomposition(per_b, lcb, "persistence")
boundary = pd.concat([bd_rg, bd_sev, bd_per], ignore_index=True)

for name, df in (("RECOVERY GAP", bd_rg), ("SEVERITY", bd_sev),
                 ("PERSISTENCE", bd_per)):
    print("\n" + name)
    display(df.drop(columns=["indicator"]))
    spread_full = df["class_mean"].max() - df["class_mean"].min()
    spread_int = df["mean_interior"].max() - df["mean_interior"].min()
    rho = stats.spearmanr(df["class_mean"], df["mean_interior"]).statistic
    note("spread of class means: {:.3f}   spread of interior means: {:.3f}"
         .format(spread_full, spread_int))
    note("rank agreement between the two orderings: Spearman {:+.3f}".format(rho))

save_table(boundary, "t06_boundary_decomposition",
           "Exact decomposition of each land-cover class mean into the share "
           "contributed by pixels clipped at the upper bound and the share "
           "contributed by pixels still varying inside (0, 1). Burned pixels "
           "only; classes with n < 500 omitted.", section=6)

## 7. Normalisation decomposition - numerator versus denominator

**Question.** Is the land-cover ordering driven by how much NBR was lost, or by the baseline the loss is divided by?

**Why.** The indicator is `clip((b - r) / |b|, 0, 1)`. Its numerator is the absolute NBR loss between the two windows; its denominator is the pre-disturbance NBR level, which is a property of the land-cover class rather than of the fire. Separating the two decides whether the class ordering says anything about disturbance at all. This is the analysis that settles the question in its strongest form.

**Data.** `nbr_raw.npy` (mean over months 0-11 and 96-107), `targets_aux.npz`, `X_scaled_meta.csv`.

**Metric.** Four variants of the same numerator, differing only in the denominator:\n\n| variant | formula |\n|---|---|\n| published | `clip(d / (|b| + eps), 0, 1)` |\n| unclipped | `d / (|b| + eps)` |\n| absolute drop | `d = b - r`, in NBR units |\n| global denominator | `clip(d / mean(|b|), 0, 1)` - one denominator for every pixel |\n| per-class denominator | `clip(d / mean(|b|) within class, 0, 1)` |\n\nPlus `Pearson(log|b|, log(d/|b|))` over pixels with a positive drop, which measures the denominator's pixel-level influence directly.

**Reading the output.** If the ordering came from disturbance, the `absolute drop` and `global denominator` columns would preserve it. If it came from the denominator, those columns will not.

**Relates to.** Thesis §4.4 and §5.4.

In [ ]:
if not HAVE_SPECTRAL:
    print("SKIPPED — nbr_raw.npy absent. See §1 provenance note.")
else:
    b = b_all[BURNED]                       # baseline: mean NBR Jan-Dec 2017
    r = r_all[BURNED]                       # recent:   mean NBR Jan-Dec 2025
    d = b - r                               # absolute loss, NBR units
    g_unclipped = d / (np.abs(b) + EPS)
    g_pub = np.clip(g_unclipped, 0, 1)

    g_global = np.clip(d / (np.abs(b).mean() + EPS), 0, 1)
    g_class = np.empty_like(g_pub)
    for code in np.unique(lcb):
        m = lcb == code
        g_class[m] = np.clip(d[m] / (np.abs(b[m]).mean() + EPS), 0, 1)

    rows = []
    for code, name in LC_NAMES.items():
        m = lcb == code
        if not m.any():
            continue
        rows.append(dict(
            land_cover=name, n=int(m.sum()),
            baseline_NBR_2017=float(b[m].mean()), recent_NBR_2025=float(r[m].mean()),
            abs_drop=float(d[m].mean()),
            published_gap=float(g_pub[m].mean()),
            unclipped_median=float(np.median(g_unclipped[m])),
            gap_global_denominator=float(g_global[m].mean()),
            gap_class_denominator=float(g_class[m].mean()),
            pct_clipped_at_1=100 * float((g_unclipped[m] >= 1).mean()),
            pct_at_or_below_0=100 * float((g_unclipped[m] <= 0).mean()),
            below_min_n=int(m.sum()) < MIN_N))
    decomposition = pd.DataFrame(rows).sort_values("published_gap", ascending=False)
    display(decomposition.drop(columns=["below_min_n"]))

    big = decomposition[~decomposition["below_min_n"]]
    print("\nClass-level rank agreement with the published ordering "
          "(Spearman, n = {} classes with n >= {}):".format(len(big), MIN_N))
    rank_rows = []
    for col in ("abs_drop", "unclipped_median", "gap_global_denominator",
                "gap_class_denominator"):
        rho = float(stats.spearmanr(big["published_gap"], big[col]).statistic)
        rank_rows.append(dict(variant=col, class_rank_spearman_vs_published=rho))
        print("   {:<26} rho = {:+.3f}".format(col, rho))

    keep = (d > 0) & (b > 0)
    r_log = float(stats.pearsonr(np.log(np.abs(b[keep])),
                                 np.log(g_unclipped[keep]))[0])
    print("\nPixel-level influence of the denominator:")
    print("   Pearson(log|baseline|, log unclipped gap) = {:+.3f}   (n = {:,})"
          .format(r_log, int(keep.sum())))
    rank_rows.append(dict(variant="pixel Pearson(log|baseline|, log unclipped gap)",
                          class_rank_spearman_vs_published=r_log))

    print("\nPixel-level agreement of each variant with the published target:")
    for label, alt in (("absolute drop", d), ("unclipped", g_unclipped),
                       ("global denominator", g_global),
                       ("per-class denominator", g_class)):
        print("   {:<24} Pearson {:+.3f}   Spearman {:+.3f}".format(
            label, float(stats.pearsonr(g_pub, alt)[0]),
            float(stats.spearmanr(g_pub, alt).statistic)))

    save_table(decomposition, "t07_normalisation_decomposition",
               "The recovery gap decomposed into numerator and denominator by "
               "land-cover class, burned pixels. 'abs_drop' is the loss in NBR "
               "units before any normalisation; 'gap_global_denominator' replaces "
               "each pixel's own baseline with the sample mean, isolating the "
               "effect of the per-pixel normalisation.", section=7)
    save_table(pd.DataFrame(rank_rows), "t07b_normalisation_rank_agreement",
               "Class-level rank agreement between the published recovery-gap "
               "ordering and each alternative denominator, plus the pixel-level "
               "correlation between the baseline and the normalised gap.",
               section=7)

In [ ]:
if HAVE_SPECTRAL:
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
    step = 40                                    # thin the scatter for legibility
    for c in SHOW:
        m = lcb == c
        if m.sum() < MIN_N:
            continue
        axes[0].scatter(b[m][::step], d[m][::step], s=2, alpha=0.25,
                        color=COLOR[c], label=LC_NAMES[c], rasterized=True)
    lim = np.linspace(0.01, float(np.percentile(b, 99)), 200)
    for lev in (0.25, 0.5, 1.0):
        axes[0].plot(lim, lev * lim, "k--", lw=0.8, alpha=0.6)
        axes[0].annotate("gap = {:g}".format(lev), (lim[-1], lev * lim[-1]),
                         fontsize=7, ha="right", va="bottom")
    axes[0].set_xlabel("baseline NBR (mean of Jan-Dec 2017)")
    axes[0].set_ylabel("absolute drop, baseline − recent")
    axes[0].set_title("The same absolute drop maps to a larger gap\n"
                      "where the baseline is small", fontsize=10)
    axes[0].legend(fontsize=7, markerscale=4)

    names = [LC_NAMES[c] for c in SHOW]
    clip_rate = [100 * float((g_unclipped[lcb == c] >= 1).mean()) for c in SHOW]
    axes[1].barh(names, clip_rate, color=[COLOR[c] for c in SHOW])
    axes[1].set_xlabel("% of burned pixels clipped at the upper bound")
    axes[1].set_title("Saturation rate by land cover", fontsize=10)
    axes[1].grid(axis="y", alpha=0)
    fig.tight_layout()
    save_figure(fig, "f02_normalisation_and_saturation",
                "Left: absolute NBR loss against baseline NBR by land cover, with "
                "iso-gap lines fanning from the origin — at a small baseline even a "
                "modest loss crosses the ceiling. Right: the resulting saturation "
                "rates.", section=7)
    plt.show()

## 8. Event-aligned post-fire NBR response, seasonality cancelled

**Question.** What does post-fire NBR recovery actually look like in each land-cover class, measured directly rather than through the indicator?

**Why.** The recovery gap is not a post-fire measure: it compares two fixed calendar windows regardless of when a pixel burned. To find out whether its class ordering reflects recovery, recovery has to be measured properly - aligned on each pixel's own fire, with the annual cycle removed. This is what settles the question.

**Data.** `nbr_raw.npy` and `y_burned.npy`. Restricted to burned pixels whose *first* documented fire leaves a full 12 months before and 24 months after inside the record (months 12-83), so every pixel contributes the same window.

**Metric.** Windows are whole 12-month blocks, so the annual cycle cancels exactly rather than approximately:\n\n```\npre = mean NBR over months [f-12, f-1]\ny1  = mean NBR over months [f+1, f+12]\ny2  = mean NBR over months [f+13, f+24]\nrelative loss = (pre - y_k) / pre        (computed where pre > 0.01)\n```\n\n`pct_still_below_pre_y2` is the share of pixels with `y2 < pre`. A negative relative loss means the pixel now exceeds its pre-fire level.

**Reading the output.** Relative loss at year 2 is the closest thing in this dataset to 'how much spectral recovery is still outstanding'. Compare its class ordering with the recovery-gap column beside it.

**Relates to.** Thesis §4.4 and §5.4.

In [ ]:
if not HAVE_SPECTRAL:
    print("SKIPPED — nbr_raw.npy absent.")
else:
    W_PRE, W_POST = 12, 24
    eligible = (first_fire >= W_PRE) & (first_fire < N_MONTHS - W_POST)
    print("burned pixels with a complete -12/+24 month window: {:,} of {:,} ({:.1f}%)"
          .format(int(eligible.sum()), N_BURNED, 100 * eligible.mean()))

    nbr_burned = np.asarray(nbr[BURNED], dtype=np.float32)

    def window_mean(arr, centres, lo, hi):
        """Mean over months [centre+lo, centre+hi-1], vectorised."""
        offs = np.arange(lo, hi)[None, :]
        idx = centres[:, None] + offs
        return arr[np.arange(len(centres))[:, None], idx].mean(axis=1)

    rows = []
    for code, name in LC_NAMES.items():
        m = eligible & (lcb == code)
        if m.sum() < 300:
            continue
        sub = nbr_burned[m]
        f = first_fire[m]
        pre = window_mean(sub, f, -W_PRE, 0)
        y1 = window_mean(sub, f, 1, 13)
        y2 = window_mean(sub, f, 13, 25)
        usable = pre > 0.01          # a relative measure needs a usable denominator
        rows.append(dict(
            land_cover=name, n=int(m.sum()),
            pre_fire_NBR=float(pre.mean()),
            abs_drop_y1=float((pre - y1).mean()),
            abs_drop_y2=float((pre - y2).mean()),
            rel_loss_y1=float(((pre - y1) / pre)[usable].mean()),
            rel_loss_y2=float(((pre - y2) / pre)[usable].mean()),
            pct_still_below_pre_y2=100 * float((y2 < pre).mean()),
            n_usable_for_relative=int(usable.sum()),
            recovery_gap_target=float(rg_b[m].mean())))
    postfire = pd.DataFrame(rows).sort_values("rel_loss_y2", ascending=False)
    display(postfire)
    note("negative relative loss means the class now exceeds its pre-fire NBR level")
    save_table(postfire, "t08_postfire_nbr_response",
               "Post-fire NBR response by land-cover class, aligned on each "
               "pixel's first documented fire, with whole 12-month windows either "
               "side so the annual cycle cancels. 'rel_loss' is the fraction of "
               "the pre-fire level not yet regained.", section=8)

In [ ]:
if HAVE_SPECTRAL:
    W = 24
    ok_ev = (first_fire >= W) & (first_fire < N_MONTHS - W)
    fig, ax = plt.subplots(figsize=(7.8, 4.6))
    for c in SHOW:
        m = ok_ev & (lcb == c)
        if m.sum() < 300:
            continue
        sub = nbr_burned[m]
        f = first_fire[m]
        idx = f[:, None] + np.arange(-W, W + 1)[None, :]
        win = sub[np.arange(len(f))[:, None], idx]
        pre = win[:, :W].mean(axis=1, keepdims=True)
        ax.plot(np.arange(-W, W + 1), (win - pre).mean(axis=0), lw=1.5,
                color=COLOR[c], label="{} (n={:,})".format(LC_NAMES[c], int(m.sum())))
    ax.axvline(0, color="crimson", lw=1.0, ls="--")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("months relative to the first documented fire")
    ax.set_ylabel("Δ NBR from the 24 months before the fire")
    ax.set_title("Fire-event-aligned NBR response and recovery, by land cover",
                 fontsize=10)
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_figure(fig, "f03_event_aligned_nbr",
                "Mean NBR relative to the 24 months preceding each pixel's first "
                "documented fire. Forest shows the deepest response and the only "
                "sustained depression; grassland returns to baseline within months. "
                "The residual 12-month oscillation is the annual cycle, which the "
                "block-averaged table in §8 removes exactly.", section=8)
    plt.show()

    # calendar-time view, absolute and relative to each class's own 2017 mean
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.2))
    months = np.arange(N_MONTHS)
    for c in SHOW:
        m = lcb == c
        if m.sum() < MIN_N:
            continue
        series = nbr_burned[m].mean(axis=0)
        axes[0].plot(months, series, lw=1.4, color=COLOR[c], label=LC_NAMES[c])
        axes[1].plot(months, series - series[:12].mean(), lw=1.4, color=COLOR[c])
    for ax, ttl, yl in ((axes[0], "Mean NBR by land cover (burned pixels)", "NBR"),
                        (axes[1], "Change from each class's own 2017 mean", "Δ NBR")):
        ax.axvspan(0, 11, color="0.85", zorder=0)
        ax.axvspan(96, 107, color="0.85", zorder=0)
        ax.set_xticks(np.arange(0, N_MONTHS, 12))
        ax.set_xticklabels(range(2017, 2026))
        ax.set_title(ttl, fontsize=10)
        ax.set_ylabel(yl)
    axes[1].axhline(0, color="k", lw=0.8)
    axes[0].legend(fontsize=8, ncol=2)
    fig.suptitle("Shaded bands are the two windows the recovery gap compares",
                 fontsize=9, y=0.02)
    fig.tight_layout()
    save_figure(fig, "f04_nbr_calendar_by_landcover",
                "Mean NBR through the record by land-cover class, burned pixels, "
                "in absolute terms and relative to each class's own 2017 mean. "
                "Shaded bands mark the two windows the recovery gap compares.",
                section=8)
    plt.show()

## 9. Does the indicator track measured recovery?

**Question.** Across land-cover classes, is the recovery-gap target correlated with the post-fire recovery measured in §8, or with its own denominator?

**Why.** §7 shows the denominator dominates the construction and §8 measures recovery properly. This puts the two side by side and produces the single number that decides how §4.4 and §5.4 must be written.

**Data.** The class-level table from §8.

**Metric.** Spearman rank correlation across land-cover classes between the recovery-gap target and each of: relative loss outstanding at year 2, the share of pixels still below their pre-fire level, the absolute drop at year 2, and the pre-fire NBR level (the indicator's own denominator). Rank correlation is used because only the ordering is claimed.\n\nComputed over **two class sets**, both reported. `all` is every class with at least 300 usable pixels. `vegetated` additionally drops **water** and **built-up**, on the stated ground that these are non-vegetated categories in which a burned label indicates land-cover misclassification or a mixed pixel rather than vegetation burning - a reading the thesis already applies to the water row of Table 11. The exclusion rule is fixed here in advance of reading the result, and both numbers are reported precisely so the choice is visible.

**Reading the output.** A correlation near zero with measured recovery, and a negative one with the pre-fire level, would mean the indicator orders classes by their baseline rather than by their recovery.\n\n**These are descriptive statistics over seven to nine points, and they are sensitive to which classes are included** - water in particular has both a low baseline and a low gap (its NBR rose), which weakens the baseline relationship when it is in. Do not lead with these numbers. The argument is carried by the pixel-level result in §7 (Pearson −0.573 over 204,503 pixels) and by the direct class comparison in §8, neither of which depends on this choice.

**Relates to.** Thesis §4.4, §5.4 and the abstract's land-cover claim.

In [ ]:
if HAVE_SPECTRAL:
    pairs = [
        ("rel_loss_y2", "relative NBR loss still outstanding at year 2"),
        ("pct_still_below_pre_y2", "% of pixels still below pre-fire NBR at year 2"),
        ("abs_drop_y2", "absolute NBR drop at year 2"),
        ("pre_fire_NBR", "pre-fire NBR level (the indicator's own denominator)"),
    ]
    # Non-vegetated classes in which a burned label indicates misclassification
    # or a mixed pixel. Fixed here before the result is read; both sets reported.
    NON_VEGETATED = {"water", "built-up"}
    CLASS_SETS = {
        "all": postfire,
        "vegetated": postfire[~postfire["land_cover"].isin(NON_VEGETATED)],
    }

    rows = []
    for set_name, df_set in CLASS_SETS.items():
        for col, label in pairs:
            rho = float(stats.spearmanr(df_set[col], df_set["recovery_gap_target"]).statistic)
            rows.append(dict(class_set=set_name, n_classes=len(df_set),
                             compared_with=label, column=col,
                             spearman_vs_recovery_gap=rho))
    indicator_vs_recovery = pd.DataFrame(rows)
    display(indicator_vs_recovery.pivot(index=["column", "compared_with"],
                                        columns="class_set",
                                        values="spearman_vs_recovery_gap").round(3))
    note("excluded from 'vegetated':", ", ".join(sorted(
        set(postfire["land_cover"]) & NON_VEGETATED)) or "none present")
    note("the two columns differ because water has both a low baseline and a low")
    note("gap (its NBR rose over the record), which weakens the baseline relation.")
    note("Lead with §7 (pixel level) and §8 (direct comparison), not with these.")
    save_table(indicator_vs_recovery, "t09_indicator_vs_measured_recovery",
               "Rank correlation across land-cover classes between the analytical "
               "recovery-gap target and directly measured post-fire NBR recovery, "
               "and between the target and its own denominator. Reported over all "
               "classes and over vegetated classes only (excluding water and "
               "built-up, where a burned label indicates misclassification). "
               "Descriptive statistics over 7-9 points; the pixel-level evidence "
               "in §7 is what carries the argument.", section=9)

    fig, ax = plt.subplots(figsize=(7.0, 4.8))
    order = postfire.sort_values("rel_loss_y2", ascending=False)
    ypos = np.arange(len(order))
    ax.barh(ypos - 0.2, order["rel_loss_y2"], height=0.38,
            color="#0D6A5F", label="measured: relative NBR loss outstanding at year 2")
    ax.barh(ypos + 0.2, order["recovery_gap_target"], height=0.38,
            color="#A2402A", label="analytical recovery-gap target")
    ax.set_yticks(ypos)
    ax.set_yticklabels(order["land_cover"])
    ax.invert_yaxis()
    ax.axvline(0, color="k", lw=0.8)
    ax.set_xlabel("fraction of pre-fire NBR level")
    ax.set_title("Measured post-fire recovery and the indicator disagree", fontsize=10)
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(axis="y", alpha=0)
    fig.tight_layout()
    save_figure(fig, "f05_indicator_vs_measured_recovery",
                "Directly measured outstanding NBR loss two years after fire "
                "(green) against the analytical recovery-gap target (red), by "
                "land-cover class. Forest has the largest outstanding loss and "
                "nearly the smallest indicator value.", section=9)
    plt.show()

## 10. Alternative formulations, including an independent index

**Question.** Does the land-cover ordering survive reasonable alternative definitions of the indicator, and does it survive changing the spectral index?

**Why.** If the ordering is robust to how the indicator is written, it is a feature of the data; if it depends on the normalisation and the bounds, it is a feature of the definition. The NDVI variant was included to test whether the ordering is specific to NBR.

**Data.** `nbr_raw.npy`, `nbr_zscore_raw.npy`, `ndvi_raw.npy`, `targets_aux.npz`.

**Metric.** Class means under six formulations, each scored against the published one by pixel-level Spearman and by class-level rank correlation. The NDVI variant applies the identical formula to a different index; the z-space variant replaces the ratio with a difference of standardised anomalies.

**Reading the output.** The NDVI row does **not** isolate index from formula: it inherits the same `÷|baseline|` normalisation, and NDVI baselines are also smaller in grassland than in forest. It shows the ordering is not specific to NBR; it does not show the ordering is ecological. §8-9 are what settle that. The z-space row should be read with the scaler statistics from §3 in mind.

**Relates to.** Thesis §4.4, §5.4.

In [ ]:
if not HAVE_SPECTRAL:
    print("SKIPPED — spectral arrays absent.")
else:
    variants = {
        "published: clip(d/|b|, 0, 1)": g_pub,
        "unclipped: d/|b|": g_unclipped,
        "absolute drop: d (NBR units)": d,
        "unclipped, winsorised at 1-99%":
            np.clip(g_unclipped, *np.percentile(g_unclipped, [1, 99])),
    }
    zb = np.nanmean(np.asarray(nbr_z[:, :12], dtype=np.float64), axis=1)[BURNED]
    zr = np.nanmean(np.asarray(nbr_z[:, -12:], dtype=np.float64), axis=1)[BURNED]
    variants["z-space difference: z_2017 - z_2025"] = zb - zr
    if HAVE_NDVI:
        ndvi = np.load(INPUTS["ndvi_raw.npy"], mmap_mode="r")
        nb_ = np.nanmean(np.asarray(ndvi[:, :12], dtype=np.float64), axis=1)[BURNED]
        nr_ = np.nanmean(np.asarray(ndvi[:, -12:], dtype=np.float64), axis=1)[BURNED]
        variants["NDVI gap, identical formula"] = np.clip(
            (nb_ - nr_) / (np.abs(nb_) + EPS), 0, 1)

    rows = []
    for label, v in variants.items():
        row = {"formulation": label,
               "pixel_spearman_vs_published":
                   float(stats.spearmanr(g_pub, v).statistic)}
        for code, name in LC_NAMES.items():
            m = lcb == code
            if m.sum() >= MIN_N:
                row[name] = float(v[m].mean())
        rows.append(row)
    alternatives = pd.DataFrame(rows)

    cls_cols = [c for c in alternatives.columns
                if c not in ("formulation", "pixel_spearman_vs_published")]
    base = alternatives.loc[0, cls_cols].astype(float)
    alternatives["class_rank_spearman_vs_published"] = [
        float(stats.spearmanr(base, alternatives.loc[i, cls_cols].astype(float)).statistic)
        for i in range(len(alternatives))]
    display(alternatives[["formulation", "pixel_spearman_vs_published",
                          "class_rank_spearman_vs_published"] + cls_cols])
    save_table(alternatives, "t10_alternative_formulations",
               "Class means of the recovery gap under alternative formulations, "
               "each scored against the published definition at pixel level and "
               "at class-ordering level. The NDVI row applies the identical "
               "formula to an independent index but inherits the same "
               "normalisation, so it tests index-specificity, not ecology.",
               section=10)

## 11. Corrected recovery-window sensitivity

**Question.** How much does the recovery gap depend on the choice of temporal window, once the comparison is made fairly?

**Why.** The published test (Notebook 06b, Experiment 9) varies three things at once. Its 6-month variant compares **Jan-Jun 2017 with Jul-Dec 2025** - opposite halves of the annual cycle - and its 24-month variant uses a baseline spanning **2017-2018**, where 2018 is the peak fire year of the record. Neither the correlations nor the conclusion drawn from them can be attributed to window length. This section varies one factor at a time.

**Data.** `nbr_raw.npy`, `y_burned.npy`. Burned pixels.

**Metric.** Four blocks:\n\n- **A - length.** Baseline fixed at Jan-Dec 2017; recent window 12, 24 or 36 months. Both ends span whole years, so seasonality cancels and only the amount of pooled recent history changes.\n- **B - sub-annual, matched.** Half-year windows drawn from the *same* calendar half at both ends, separating 'does a full annual cycle matter' from 'are the ends aligned'.\n- **C - baseline.** Extending the baseline to 2017-2018, reported as fire-contaminated. The record starts in January 2017, so the baseline cannot be lengthened backwards: 2017 is the only uncontaminated pre-disturbance reference available, which is a structural limitation of the study period rather than a tuning choice.\n- **D - control.** The published 6-month configuration, reproduced so the size of the seasonal confound can be stated.\n\nEach variant is scored against the published 12-month target by Pearson, Spearman, and the Jaccard overlap of the top-decile selection.

**Reading the output.** Correlation measures whether the values agree; **Jaccard of the top 10% measures whether the same pixels would be prioritised**, which is what a prioritisation framework actually uses and is the more demanding test.

**Relates to.** Thesis §4.9 and Table 16.

In [ ]:
if not HAVE_SPECTRAL:
    print("SKIPPED — nbr_raw.npy absent.")
else:
    nbr_all = np.asarray(nbr, dtype=np.float32)

    def gap_for_windows(base_slice, recent_slice):
        bb = np.nanmean(nbr_all[:, base_slice].astype(np.float64), axis=1)
        rr = np.nanmean(nbr_all[:, recent_slice].astype(np.float64), axis=1)
        gg = np.zeros(N_PIX, dtype=np.float32)
        gg[BURNED] = np.clip((bb - rr) / (np.abs(bb) + EPS), 0, 1)[BURNED]
        return gg[BURNED]

    CONFIGS = [
        ("A: recent 12 mo (2025) — published", slice(0, 12), slice(96, 108), "length"),
        ("A: recent 24 mo (2024-2025)",        slice(0, 12), slice(84, 108), "length"),
        ("A: recent 36 mo (2023-2025)",        slice(0, 12), slice(72, 108), "length"),
        ("B: Jan-Jun 2017 vs Jan-Jun 2025",    slice(0, 6),  slice(96, 102), "sub-annual, matched"),
        ("B: Jul-Dec 2017 vs Jul-Dec 2025",    slice(6, 12), slice(102, 108), "sub-annual, matched"),
        ("C: baseline 2017-2018 (contaminated)", slice(0, 24), slice(84, 108), "baseline"),
        ("D: published 6-month (Jan-Jun 17 vs Jul-Dec 25)", slice(0, 6), slice(102, 108), "confounded"),
    ]

    ref = gap_for_windows(slice(0, 12), slice(96, 108))
    assert np.corrcoef(ref, rg_b)[0, 1] > 0.9999, "block A row 1 must equal the published target"
    k = int(0.10 * len(ref))
    top_ref = set(np.argsort(ref)[-k:])

    rows = []
    for label, bs, rs, factor in CONFIGS:
        v = gap_for_windows(bs, rs)
        top_v = set(np.argsort(v)[-k:])
        rows.append(dict(
            configuration=label, factor_varied=factor, mean=float(v.mean()),
            pearson_vs_published=float(stats.pearsonr(ref, v)[0]),
            spearman_vs_published=float(stats.spearmanr(ref, v).statistic),
            jaccard_top10pct=len(top_ref & top_v) / len(top_ref | top_v)))
    windows = pd.DataFrame(rows)
    display(windows)
    note("top-decile Jaccard is the operationally relevant number: it is the share")
    note("of prioritised pixels that two definitions agree on.")
    save_table(windows, "t11_window_sensitivity_corrected",
               "Recovery-window sensitivity with one factor varied at a time. "
               "Block A varies recent-window length with the baseline fixed and "
               "both ends seasonally complete; block B uses matched calendar "
               "halves; block C extends the baseline into the 2018 fire year; row "
               "D reproduces the published configuration, whose distance from "
               "block B measures the seasonal confound.", section=11)

    fig, ax = plt.subplots(figsize=(8.6, 4.2))
    ypos = np.arange(len(windows))
    ax.barh(ypos - 0.2, windows["spearman_vs_published"], height=0.38,
            color="#0D6A5F", label="Spearman with the published target")
    ax.barh(ypos + 0.2, windows["jaccard_top10pct"], height=0.38,
            color="#A2402A", label="Jaccard overlap of the top decile")
    ax.set_yticks(ypos)
    ax.set_yticklabels(windows["configuration"], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.02)
    ax.set_xlabel("agreement with the published 12-month definition")
    ax.set_title("Value agreement holds up far better than the priority set",
                 fontsize=10)
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(axis="y", alpha=0)
    fig.tight_layout()
    save_figure(fig, "f06_window_sensitivity",
                "Sensitivity of the recovery gap to the temporal window, one "
                "factor varied at a time. Rank correlation stays moderate while "
                "the top-decile selection overlaps by only about a quarter.",
                section=11)
    plt.show()

## 12. Persistence threshold sensitivity

**Question.** Does the hard-coded `z < -0.5` in the persistence definition matter?

**Why.** The threshold appears in notebook 04 as a bare constant with no citation, so it is a design choice rather than a literature-derived value. What matters is therefore not a retrofitted citation but a demonstration of how much the choice affects the result.

**Data.** `nbr_zscore_raw.npy`, `y_burned.npy`, `targets_aux.npz`.

**Metric.** Persistence recomputed at z < -0.25, -0.5, -0.75 and -1.0, keeping the definition otherwise identical: the fraction of months from the last fire month onward below the threshold, and zero where fewer than two months remain. Each is scored against the published version by Pearson and Spearman, with class means reported.

**Reading the output.** High correlations and a stable class ordering across a fourfold range of the threshold would mean the constant is undocumented but inconsequential - a stronger answer than a citation.

**Relates to.** Thesis §3.2.7 and §4.9.

In [ ]:
if not HAVE_SPECTRAL:
    print("SKIPPED — nbr_zscore_raw.npy absent.")
else:
    z_burn = np.asarray(nbr_z[BURNED], dtype=np.float32)
    idx_m_b = np.arange(N_MONTHS)[None, :]
    post_mask = idx_m_b >= last_fire_b[:, None]
    n_post_b = post_mask.sum(axis=1)

    rows = []
    for thr in (-0.25, -0.5, -0.75, -1.0):
        vals = np.where(n_post_b > 1,
                        ((z_burn < thr) & post_mask).sum(axis=1) / np.maximum(n_post_b, 1),
                        0.0).astype(np.float32)
        row = dict(threshold=thr, mean=float(vals.mean()),
                   pearson_vs_published=float(stats.pearsonr(per_b, vals)[0]),
                   spearman_vs_published=float(stats.spearmanr(per_b, vals).statistic))
        for code, name in LC_NAMES.items():
            m = lcb == code
            if m.sum() >= MIN_N:
                row[name] = float(vals[m].mean())
        rows.append(row)
    persistence_sens = pd.DataFrame(rows)
    display(persistence_sens)

    published = persistence_sens.loc[persistence_sens["threshold"] == -0.5].iloc[0]
    r_pub = float(published["pearson_vs_published"])
    assert r_pub > 0.999, "the -0.5 row must reproduce the stored persistence target"
    note("the -0.5 row reproduces the stored target at r = {:.7f}".format(r_pub))
    note("it is not exactly 1 because reconstructing the z-score channel from")
    note("standardised float32 carries an error of roughly 0.007 z-units (its scale")
    note("is 72,871 — see §3), so pixels within that distance of the threshold can")
    note("flip. This is a further consequence of the n=2 baseline sigma, and it")
    note("affects the stored severity and persistence targets themselves, not just")
    note("this recomputation.")
    save_table(persistence_sens, "t12_persistence_threshold_sensitivity",
               "Persistence recomputed at four z-score thresholds. The published "
               "value is -0.5, which appears in Notebook 04 as a constant with no "
               "recorded derivation.", section=12)
    del z_burn, post_mask

## 13. Model versus land cover - three explanations separated

**Question.** Is the model's high agreement with the analytical targets explained by it using land-cover identity as a shortcut?

**Why.** One hypothesis is that the model reproduces patterns already encoded in the target and in the land-cover input rather than learning anything about recovery. That splits into three claims that need separating: (a) land cover explains between-class structure in the target; (b) the model exploits class identity as a shortcut; (c) the model reconstructs the target from the temporal spectral inputs. They are not the same claim and they have different tests.

**Data.** `predictions_test.npz` (per-pixel model outputs on the 15% test split), `targets_aux.npz`, `X_scaled_meta.csv`. Burned test pixels only - the population the regression heads received gradient on under `MaskedMultiTaskLoss`.

**Metric.** Three tests, one per claim:\n\n- **(a)** η² = between-class sum of squares / total, i.e. the share of target variance explained by class membership. ω² is reported alongside because η² is biased upward by the number of groups; agreement between them shows the bias is negligible here.\n- **(b)** Nested OLS. `R²(target ~ model)` versus `R²(target ~ model + LC dummies)`; the increment is what class identity adds *given* the model's own prediction. Plus the within-class correlation, with class means removed from target and prediction alike, so class identity cannot contribute to it. A percentile bootstrap over pixels gives the interval.\n- **(c)** is not a statistical test but a structural fact, stated in the markdown below the cell.

**Reading the output.** An incremental R² near zero means the model already carries whatever land-cover signal exists. A high within-class correlation means the model tracks variation that class identity cannot explain. Together they refute the shortcut hypothesis.

**Relates to.** Thesis §4.3, §5.1 and §5.4.

In [ ]:
if not HAVE_PREDS:
    print("SKIPPED — predictions_test.npz absent.")
else:
    preds = np.load(INPUTS["predictions_test.npz"])
    idx_test = preds["indices"]
    burned_test = targets["recurrence"][idx_test] > 0
    lc_test = LC_ALL[idx_test][burned_test]
    codes_t = np.unique(lc_test)
    dummies = (lc_test[:, None] == codes_t[None, :]).astype(float)[:, 1:]
    print("test pixels: {:,}   burned test pixels: {:,}"
          .format(len(idx_test), int(burned_test.sum())))

    def ols_r2(y, X):
        X = np.column_stack([np.ones(len(y)), X])
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        res = y - X @ beta
        return 1.0 - float(res @ res) / float((y - y.mean()) @ (y - y.mean()))

    def eta2_omega2(y, groups):
        grand = y.mean()
        ssb = ssw = 0.0
        k = 0
        for c in np.unique(groups):
            v = y[groups == c]
            k += 1
            ssb += len(v) * (v.mean() - grand) ** 2
            ssw += float(((v - v.mean()) ** 2).sum())
        sst = ssb + ssw
        msw = ssw / (len(y) - k)
        return ssb / sst, (ssb - (k - 1) * msw) / (sst + msw)

    rows = []
    for name in ("recovery_gap", "severity", "persistence"):
        y = targets[name][idx_test][burned_test]
        yh = preds[name][burned_test]
        e2, w2 = eta2_omega2(y, lc_test)
        r2_lc = ols_r2(y, dummies)
        r2_m = ols_r2(y, yh)
        r2_both = ols_r2(y, np.column_stack([yh, dummies]))

        yc, yhc = y.copy(), yh.copy()
        for c in codes_t:
            m = lc_test == c
            yc[m] -= y[m].mean()
            yhc[m] -= yh[m].mean()
        boots = [float(stats.pearsonr(yc[i], yhc[i])[0])
                 for i in (RNG.integers(0, len(yc), len(yc)) for _ in range(400))]
        lo, hi = np.percentile(boots, [2.5, 97.5])
        rows.append(dict(
            indicator=name, n=int(burned_test.sum()),
            eta2_landcover=e2, omega2_landcover=w2,
            r2_landcover_only=r2_lc, r2_model_only=r2_m, r2_model_plus_lc=r2_both,
            incremental_r2_of_lc_given_model=r2_both - r2_m,
            incremental_r2_of_model_given_lc=r2_both - r2_lc,
            within_class_pearson=float(stats.pearsonr(yc, yhc)[0]),
            within_class_ci_lo=float(lo), within_class_ci_hi=float(hi),
            within_class_spearman=float(stats.spearmanr(yc, yhc).statistic)))
    model_vs_lc = pd.DataFrame(rows)
    display(model_vs_lc)
    save_table(model_vs_lc, "t13_model_vs_landcover",
               "Three competing explanations separated, on burned test pixels. "
               "'incremental_r2_of_lc_given_model' is what land-cover dummies add "
               "on top of the model's own prediction; near zero means the model "
               "already carries any land-cover signal. 'within_class_pearson' "
               "removes class means from target and prediction alike, so it "
               "measures agreement class identity cannot explain.", section=13)

**On explanation (c).** No statistical test can settle it, because it is a
structural fact about the target rather than a hypothesis about the model:

```
recovery_gap = clip((mean NBR[months 0:12] - mean NBR[months 96:108]) / |mean NBR[0:12]|, 0, 1)
```

and `NBR` is feature 8 of the 52 the model receives at every one of the 108
months. The target is a deterministic function of two windows of a single input
channel, so the reconstruction ceiling is 1.0 by construction. An R² of 0.965
measures how closely the encoder approximates that arithmetic - a real
capability, and not evidence about ecology. The same fact explains the Table 13
result that Random Forest on 104 temporal summary statistics beats all three
sequence models on these targets: the targets *are* temporal summary statistics.

This is the point to state plainly in §4.3 and §5.1, and it is stronger than the
hedged version currently written.

## 14. Where the reconstruction R² comes from

**Question.** Is the model's reconstruction accuracy driven by the pixels at the bounds?

**Why.** §6 shows the targets are strongly bimodal. Predicting a value that is exactly 0 or exactly 1 for half the sample is an easier problem than predicting a graded value, and the two are pooled in the headline R². Splitting them shows how much of the reported accuracy is graded reconstruction.

**Data.** `predictions_test.npz`, `targets_aux.npz`. Burned test pixels.

**Metric.** R², Pearson and Spearman computed twice: over all burned test pixels, and over the interior subset with 0 < target < 1. The interior figure is a conditional statistic, not a counterfactual - it answers 'how well does the model reconstruct the graded part', not 'what would R² be without bounds'.

**Reading the output.** A large drop from full to interior means the headline number is carried by boundary mass.

**Relates to.** Thesis Table 9 and §4.3.

In [ ]:
if HAVE_PREDS:
    rows = []
    for name in ("recovery_gap", "severity", "persistence"):
        y = targets[name][idx_test][burned_test]
        yh = preds[name][burned_test]
        interior = (y > EPS) & (y < 1 - EPS)

        def _r2(a, b_):
            return 1.0 - float(((a - b_) ** 2).sum()) / float(((a - a.mean()) ** 2).sum())

        rows.append(dict(
            indicator=name, n_all=int(len(y)),
            pct_at_bounds=100 * float((~interior).mean()),
            r2_all=_r2(y, yh),
            pearson_all=float(stats.pearsonr(y, yh)[0]),
            n_interior=int(interior.sum()),
            r2_interior=_r2(y[interior], yh[interior]),
            pearson_interior=float(stats.pearsonr(y[interior], yh[interior])[0]),
            spearman_interior=float(stats.spearmanr(y[interior], yh[interior]).statistic)))
    interior_r2 = pd.DataFrame(rows)
    display(interior_r2)
    save_table(interior_r2, "t14_interior_vs_boundary_r2",
               "Reconstruction accuracy on all burned test pixels and on the "
               "interior subset with 0 < target < 1, showing how much of each "
               "headline figure is carried by pixels at the bounds.", section=14)

## 15. Classification metrics under class imbalance

**Question.** How good is fire detection at the prevalence it actually faces?

**Why.** ROC-AUC stays high under extreme imbalance because it is dominated by the abundant true negatives. At a positive rate of 0.53% the precision-recall equivalent is the metric that can fall apart under imbalance, and therefore the one worth reporting alongside it.

**Data.** `predictions_test.npz`: per-pixel-month probabilities, labels and the validity mask, restricted to valid observations.

**Metric.** ROC-AUC, average precision (PR-AUC), and the no-skill PR baseline, which for a precision-recall curve is the positive prevalence itself. Lift is PR-AUC divided by that baseline. The F1-optimal point on the curve is reported beside the operating threshold actually used, to show how close the deployed threshold is to optimal.

**Reading the output.** PR-AUC far above prevalence means the ranking is genuinely informative and not an artefact of imbalance. This is a good result and should be reported beside ROC-AUC rather than instead of it.

**Relates to.** Thesis §4.2, Table 7 and Table 8.

In [ ]:
if HAVE_PREDS:
    from sklearn.metrics import (average_precision_score, precision_recall_curve,
                                 roc_auc_score)
    valid = preds["mask"].astype(bool)
    y_true = preds["burned_labels"][valid]
    y_score = preds["burned_probs"][valid]
    prevalence = float(y_true.mean())
    roc = float(roc_auc_score(y_true, y_score))
    ap = float(average_precision_score(y_true, y_score))

    prec, rec, thr = precision_recall_curve(y_true, y_score)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    best = int(np.nanargmax(f1))
    at_op = int(np.searchsorted(thr, BURN_THRESHOLD))

    clf = pd.DataFrame([dict(
        n_pixel_months=int(valid.sum()), n_positive=int(y_true.sum()),
        positive_rate=prevalence, roc_auc=roc, pr_auc=ap,
        pr_auc_no_skill_baseline=prevalence, pr_auc_lift=ap / prevalence,
        operating_threshold=BURN_THRESHOLD,
        precision_at_operating=float(prec[at_op]), recall_at_operating=float(rec[at_op]),
        f1_at_operating=float(f1[at_op]),
        best_f1_on_curve=float(f1[best]), best_f1_threshold=float(thr[best]))])
    display(clf.T.rename(columns={0: "value"}))
    note("PR-AUC {:.4f} against a no-skill baseline of {:.5f} — a {:.0f}x lift"
         .format(ap, prevalence, ap / prevalence))
    save_table(clf, "t15_classification_metrics",
               "Fire-detection metrics on the held-out test set at the natural "
               "pixel-month positive rate, including PR-AUC and its no-skill "
               "baseline (the prevalence).", section=15)

    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    ax.plot(rec, prec, lw=1.6, color="#0D6A5F")
    ax.axhline(prevalence, color="#A2402A", ls="--", lw=1.0,
               label="no-skill baseline = prevalence ({:.3%})".format(prevalence))
    ax.scatter([rec[at_op]], [prec[at_op]], s=42, zorder=5, color="#A2402A",
               label="operating point (threshold {:.3f})".format(BURN_THRESHOLD))
    ax.set_xlabel("recall")
    ax.set_ylabel("precision")
    ax.set_title("Precision-recall, fire detection\nPR-AUC = {:.3f}".format(ap),
                 fontsize=10)
    ax.legend(fontsize=7, loc="upper right")
    fig.tight_layout()
    save_figure(fig, "f07_precision_recall",
                "Precision-recall curve for per-pixel-month fire detection on the "
                "held-out test set, with the no-skill baseline and the deployed "
                "operating point marked.", section=15)
    plt.show()

## 16. Display-mask semantics and the severity criterion

**Question.** What do the two thresholds used in the Results chapter actually select?

**Why.** Two separate problems. First, the indicator maps in Figures 10-11 are masked by `burned_prob > 0.502`, and the thesis presents that beside a per-month precision of 0.714 - but the raster stores the *maximum* over 108 monthly probabilities, so the mask means 'flagged in at least one month' and its landscape-level precision is a different number. Second, Table 19 selects pixels on `severity ≥ 0.5`, an output the regression head received no training signal for on unburned pixels. Both need to be stated explicitly.

**Data.** The 30 m rasters in `outputs/inference_full_roi/`, and `predictions_test.npz` for the pixel-level rates.

**Metric.** Cross-tabulation over eligible ROI pixels. `burned_prob > 0.502` is compared against `model_recurrence ≥ 0.2` (equivalently, count of months above threshold ≥ 1) to establish that they are the same set. The severity criterion is cross-tabulated against both the burn mask and the documented fire raster, with the Jaccard index between the two criteria. Landscape precision is then estimated from the test-set detection rates applied to ROI prevalence.

**Reading the output.** Precision here is measured against the fire-reference inventory, which is incomplete; it is precision *relative to available reference data*, not an absolute error rate. The point is the gap between it and the number the thesis quotes alongside these maps.

**Relates to.** Thesis §3.4.6, §4.11, Table 19, §5.5.

In [ ]:
if not HAVE_RASTERS:
    print("SKIPPED — 30 m rasters absent.")
else:
    try:
        import rasterio
    except ImportError:
        rasterio = None
        print("SKIPPED — rasterio not installed.")

if HAVE_RASTERS and rasterio is not None:
    def read_band(name):
        with rasterio.open(os.path.join(RAST, name)) as src:
            return src.read(1), src.profile

    bp, prof = read_band("burned_prob_30m.tif")
    mr, _ = read_band("model_recurrence_30m.tif")
    sv, _ = read_band("severity_30m.tif")
    gt, _ = read_band("recurrence_30m.tif")

    elig = ~np.isnan(bp)
    n_elig = int(elig.sum())
    m_bp = elig & (bp > BURN_THRESHOLD)
    m_cnt = elig & (mr >= 0.2 - 1e-6)        # count>=1 since mr = clip(count/5, 0, 1)
    m_gt = elig & (gt > 0)
    m_sv = elig & (sv >= 0.5)

    agreement = float((m_bp == m_cnt)[elig].mean())
    rows = [
        dict(quantity="eligible 30 m pixels", count=n_elig, pct_of_eligible=100.0),
        dict(quantity="burned_prob > 0.502 (the display mask)",
             count=int(m_bp.sum()), pct_of_eligible=100 * m_bp.sum() / n_elig),
        dict(quantity="model_recurrence >= 1 month",
             count=int(m_cnt.sum()), pct_of_eligible=100 * m_cnt.sum() / n_elig),
        dict(quantity="documented burned (merged inventory)",
             count=int(m_gt.sum()), pct_of_eligible=100 * m_gt.sum() / n_elig),
        dict(quantity="severity >= 0.5 (the Table 19 criterion)",
             count=int(m_sv.sum()), pct_of_eligible=100 * m_sv.sum() / n_elig),
        dict(quantity="severity >= 0.5 AND burned_prob > 0.502",
             count=int((m_sv & m_bp).sum()),
             pct_of_eligible=100 * (m_sv & m_bp).sum() / n_elig),
        dict(quantity="severity >= 0.5 AND NOT flagged as burned",
             count=int((m_sv & ~m_bp).sum()),
             pct_of_eligible=100 * (m_sv & ~m_bp).sum() / n_elig),
        dict(quantity="severity >= 0.5 AND documented burned",
             count=int((m_sv & m_gt).sum()),
             pct_of_eligible=100 * (m_sv & m_gt).sum() / n_elig),
    ]
    masks = pd.DataFrame(rows)
    display(masks)
    jac = (m_sv & m_bp).sum() / (m_sv | m_bp).sum()
    print("\nburned_prob mask vs model_recurrence>=1 : identical on {:.4f}% of eligible pixels"
          .format(100 * agreement))
    print("severity>=0.5 vs burned_prob>0.502      : Jaccard = {:.3f}".format(jac))
    print("share of the severity>=0.5 set the model does NOT call burned: {:.1f}%"
          .format(100 * (m_sv & ~m_bp).sum() / m_sv.sum()))
    save_table(masks, "t16_mask_semantics",
               "What the two thresholds used in the Results chapter select across "
               "the eligible ROI at 30 m.", section=16)

if HAVE_PREDS:
    # Pixel-level detection rates behind the max-over-months rule.
    probs_2d = preds["burned_probs"].reshape(len(idx_test), -1)
    mask_2d = preds["mask"].reshape(len(idx_test), -1)
    labs_2d = preds["burned_labels"].reshape(len(idx_test), -1)
    never = targets["recurrence"][idx_test] == 0
    flagged_month = (probs_2d > BURN_THRESHOLD) & mask_2d
    flagged_pixel = flagged_month.any(axis=1)

    per_month_fpr = float(flagged_month[never].sum() / mask_2d[never].sum())
    fpr_pixel = float(flagged_pixel[never].mean())
    tpr_pixel = float(flagged_pixel[~never].mean())
    ROI_PREVALENCE = 0.0180        # burned share of the eligible ROI (exp8/results.json)
    exp_flag = ROI_PREVALENCE * tpr_pixel + (1 - ROI_PREVALENCE) * fpr_pixel
    implied_precision = ROI_PREVALENCE * tpr_pixel / exp_flag

    maxrule = pd.DataFrame([dict(
        per_month_false_positive_rate_on_never_burned=per_month_fpr,
        pct_never_burned_flagged_in_at_least_one_month=100 * fpr_pixel,
        pct_burned_flagged_in_at_least_one_month=100 * tpr_pixel,
        pixel_precision_on_balanced_test_set=float(
            (flagged_pixel & ~never).sum() / flagged_pixel.sum()),
        roi_burned_prevalence=ROI_PREVALENCE,
        expected_flagged_share_of_roi=exp_flag,
        implied_landscape_precision_vs_inventory=implied_precision)])
    display(maxrule.T.rename(columns={0: "value"}))
    note("The balanced test set is ~50% burned, so its pixel-level precision is")
    note("optimistic. Re-weighting the same rates to ROI prevalence gives the")
    note("landscape figure, which is what the maps in Figures 10-11 represent.")
    save_table(maxrule, "t16b_max_over_months_rule",
               "Detection rates behind the max-over-months display mask, and the "
               "landscape-level precision implied when they are re-weighted to the "
               "ROI's burned prevalence. Precision is relative to the incomplete "
               "fire-reference inventory.", section=16)

## 17. Which land-cover definition does Table 11 use?

**Question.** Do the two per-land-cover breakdowns in the thesis use the same class assignment?

**Why.** Table 11 was regenerated by a separate notebook that reads the land-cover one-hot at month 0 - the 2018 map used as a proxy for 2017. `metrics.per_lc_stats`, which produced the `per_lc` block of `results_main.json`, uses `lc_dominant`, the modal class across all 108 months. If the two disagree the thesis is presenting two different stratifications under one name.

**Data.** `targets_aux.npz`, `X_scaled_meta.csv`, `predictions_test.npz`, and the class counts reported in the thesis Table 11.

**Metric.** Reproduce Table 11 using `lc_dominant` and compare the per-class n against the published counts. Totals must match by construction; per-class counts will not if the definitions differ.

**Reading the output.** A total that matches with per-class counts that do not is proof the two tables use different land-cover definitions. Fallow is the most affected class and is one of the two highest recovery-gap classes, so the choice matters for §5.4.

**Relates to.** Thesis Table 11 and §4.4.

In [ ]:
if HAVE_PREDS:
    THESIS_TABLE11_N = {"forest": 5561, "orchards": 554, "shrubland": 9652,
                        "grassland": 18325, "cropland": 7308,
                        "covered agriculture": 45, "built-up": 114, "bare": 895,
                        "fallow": 1608, "water": 241}
    rows = []
    for code, name in LC_NAMES.items():
        m = burned_test & (LC_ALL[idx_test] == code)
        if not m.any():
            continue
        rows.append(dict(
            land_cover=name, n_lc_dominant=int(m.sum()),
            n_thesis_table_11=THESIS_TABLE11_N.get(name),
            sev_target=float(targets["severity"][idx_test][m].mean()),
            sev_pred=float(preds["severity"][m].mean()),
            per_target=float(targets["persistence"][idx_test][m].mean()),
            per_pred=float(preds["persistence"][m].mean()),
            rg_target=float(targets["recovery_gap"][idx_test][m].mean()),
            rg_pred=float(preds["recovery_gap"][m].mean())))
    lc_check = pd.DataFrame(rows)
    lc_check["n_difference"] = lc_check["n_lc_dominant"] - lc_check["n_thesis_table_11"]
    display(lc_check)
    print("total (lc_dominant): {:,}   total (thesis Table 11): {:,}".format(
        int(lc_check["n_lc_dominant"].sum()), int(lc_check["n_thesis_table_11"].sum())))
    note("totals agree by construction; per-class counts differ, which shows the")
    note("two tables stratify by different land-cover definitions")
    save_table(lc_check, "t17_table11_landcover_definition_check",
               "Thesis Table 11 reproduced using lc_dominant (the modal class over "
               "108 months) and compared against the published per-class counts, "
               "which come from the current-year one-hot at month 0.", section=17)

    # Thesis-ready replacement for Table 11, on the resolved definition.
    revised = lc_check[["land_cover", "n_lc_dominant", "sev_target", "sev_pred",
                        "per_target", "per_pred", "rg_target", "rg_pred"]].copy()
    revised = revised.rename(columns={"n_lc_dominant": "n"})
    revised["burned_prob_mean"] = [
        float(preds["burned_probs"].reshape(len(idx_test), -1)[
            burned_test & (LC_ALL[idx_test] == code)].max(axis=1).mean())
        for code in LC_NAMES if (burned_test & (LC_ALL[idx_test] == code)).any()]
    for a, b_ in (("sev", "severity"), ("per", "persistence"), ("rg", "recovery gap")):
        revised["abs_diff_" + a] = (revised[a + "_pred"] - revised[a + "_target"]).abs()
    revised["reliable"] = revised["n"] >= MIN_N
    display(revised)
    big = revised[revised["reliable"]]
    note("classes with n >= {}: max |estimate - target| = severity {:.3f}, "
         "persistence {:.3f}, recovery gap {:.3f}".format(
             MIN_N, big["abs_diff_sev"].max(), big["abs_diff_per"].max(),
             big["abs_diff_rg"].max()))
    save_table(revised, "t17b_table11_revised",
               "Replacement for thesis Table 11, stratified by lc_dominant (the "
               "modal current-year land-cover class across the 108 months). This "
               "definition is reproducible from X_scaled_meta.csv and is the one "
               "metrics.per_lc_stats uses, so this table and the per_lc block of "
               "results_main.json now agree. 'burned_prob_mean' is the mean "
               "maximum monthly fire probability on these pixels.", section=17)

## 18. Result registry

Every table and figure this notebook writes, with the section that produced it
and the thesis destination it is intended for. Use this when revising: each
claim in the thesis should trace to a row here.

Regenerate everything with **Kernel → Restart & Run All**. Files are overwritten
in place, so the registry stays accurate.

In [ ]:
registry = pd.DataFrame(REGISTRY)

# Thesis destination for each output. Anything not listed here is diagnostic
# support rather than a direct thesis result.
DESTINATION = {
    "nb08_t00_input_inventory": "data statement / appendix",
    "nb08_t01_extraction_verification": "appendix — reproducibility",
    "nb08_t02_code_provenance_checks": "methods corrections (§3.2.2, §3.2.7, §3.4.6)",
    "nb08_t03_baseline_audit": "§3.2.7 and §5.7 limitations",
    "nb08_t03b_scaler_statistics": "§5.7 limitations (n=2 baseline sigma, C308)",
    "nb08_t04_targets_by_landcover": "§4.4 — replaces the land-cover means table",
    "nb08_f01_targets_by_landcover": "§4.4 — new figure",
    "nb08_t05_recovery_gap_by_lc_and_fire_year": "§4.4 — fire-timing control",
    "nb08_t06_boundary_decomposition": "§4.4 — saturation rates beside every class mean",
    "nb08_t07_normalisation_decomposition": "§4.4 / §5.4 — the central new result",
    "nb08_t07b_normalisation_rank_agreement": "§5.4 — supporting statistics",
    "nb08_f02_normalisation_and_saturation": "§5.4 — new figure",
    "nb08_t08_postfire_nbr_response": "§4.4 / §5.4 — measured recovery",
    "nb08_f03_event_aligned_nbr": "§4.4 — the figure the supervisor asked for",
    "nb08_f04_nbr_calendar_by_landcover": "§4.4 — supporting figure",
    "nb08_t09_indicator_vs_measured_recovery": "§5.4 — supporting correlation; report both class sets",
    "nb08_f05_indicator_vs_measured_recovery": "§5.4 — new figure",
    "nb08_t10_alternative_formulations": "§4.9 sensitivity / §5.7 limitations",
    "nb08_t11_window_sensitivity_corrected": "§4.9 — replaces Table 16",
    "nb08_f06_window_sensitivity": "§4.9 — new figure",
    "nb08_t12_persistence_threshold_sensitivity": "§4.9 — answers C409",
    "nb08_t13_model_vs_landcover": "§4.3 / §5.1 / §5.4",
    "nb08_t14_interior_vs_boundary_r2": "§4.3 — qualifies Table 9",
    "nb08_t15_classification_metrics": "§4.2 — adds PR-AUC to Table 7 (C729)",
    "nb08_f07_precision_recall": "§4.2 — new figure",
    "nb08_t16_mask_semantics": "§4.11 / §5.6 — replaces the Table 19 basis",
    "nb08_t16b_max_over_months_rule": "§3.4.6 / §4.11 — mask disclosure",
    "nb08_t17_table11_landcover_definition_check": "§4.4 footnote — land-cover definition audit",
    "nb08_t17b_table11_revised": "§4.4 — replaces Table 11",
}
registry["thesis_destination"] = registry["stem"].map(DESTINATION).fillna("(diagnostic)")
registry = registry[["section", "kind", "stem", "thesis_destination", "files", "caption"]]
registry = registry.sort_values(["section", "kind", "stem"], key=lambda s: s.astype(str))

pd.set_option("display.max_colwidth", 45)
display(registry[["section", "kind", "stem", "thesis_destination"]])

reg_path = os.path.join(TAB, PREFIX + "t18_result_registry.csv")
registry.to_csv(reg_path, index=False)
with open(os.path.join(TAB, PREFIX + "t18_result_registry.md"), "w") as fh:
    fh.write("# Result registry — notebooks/08_target_diagnostics.ipynb\n\n")
    fh.write("Generated " + RUN_STAMP + ". Thesis result -> notebook section -> output file.\n\n")
    fh.write(_markdown_table(registry[["section", "kind", "stem",
                                       "thesis_destination", "files"]], fmt="{}"))
print("\n-> outputs/tables/" + PREFIX + "t18_result_registry.csv  (+ .md)")
print("outputs written:", len(registry))

---

## Provenance summary

| Layer | Produced by | Consumed here |
|---|---|---|
| Sentinel-2 composites, baselines, land cover, fire labels | Notebook 01 (GEE) | indirectly, via the extracted arrays |
| Per-pixel time series, analytical targets, splits, scaler | Notebook 04 | `targets_aux.npz`, `y_burned.npy`, `X_scaled_meta.csv`, `splits.json` |
| Trained model, test predictions | Notebook 05 | `predictions_test.npz`, `results_main.json` |
| Full-ROI inference rasters at 30 m | Notebook 07b | `outputs/inference_full_roi/*.tif` |
| Raw NBR / NBR z-score / NDVI columns | `extract_nbr_arrays.py` or `stream_extract_nbr.py` | `nbr_raw.npy`, `nbr_zscore_raw.npy`, `ndvi_raw.npy` |
| **The diagnostic tables and figures listed in the registry** | **this notebook** | `outputs/tables/nb08_*`, `outputs/figures/nb08_*` |

## Notes for the write-up

1. **`rg_mean_interior` and `r2_interior` are conditional statistics, not
   counterfactuals.** They describe the subset still varying inside the bounds.
   They must not be reported as "what the value would be without clipping".
2. **The class-level rank correlations in §9 use nine classes.** They are
   descriptive; the pixel-level evidence in §7 is what carries the argument.
3. **Precision in §16 is measured against an incomplete inventory.** Report it as
   precision relative to available reference data.
4. **Nothing here licenses an ecological claim.** The pipeline observes no
   vegetation composition, biomass, structure or function. §8 measures *spectral*
   recovery, which is the strongest available statement.
